# MNIST Neural Network

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Importing Dataset

In [2]:
data = pd.read_csv("train.csv")

In [3]:
data.head()

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Formatting Data

In [4]:
data = np.array(data)

#Getting dimensions of the data
#42000 rows, 785 cols -> 1 label + 784 pixels
m,n = data.shape

#For training shuffling the data
np.random.shuffle(data)

In [5]:
print(m,n)

42000 785


### Important Terminologies
We use separate datasets to learn, tune, and evaluate a model.

**Train set:** 
- The model learns digit patterns by updating weights using backpropagation
   
**Dev (validation) set:**
- Used to tune hyperparameters and training decisions
  
**Test set:**
- Used once at the end for final performance measurement.
- Here the labels are hidden, so Kaggle evaluates the predictions for us

This separation prevents overfitting and ensures reliable generalization.

So just a dev/train split is needed

In [6]:
#Creating dev/train split to check for overfitting
#Following 80% train 20% dev split
split = int(0.2 * m)
data_dev = data[0:split]

#Transposing the data
data_dev = data_dev.T
#Label values
Y_dev = data_dev[0]
#Remove the label column only pixels
X_dev = data_dev[1:n]

In [7]:
data_train = data[split:m]

#Transposing the data
data_train = data_train.T
#Label values
Y_train = data_train[0]
#Remove the label column only pixels
X_train = data_train[1:n]

### X Y split explained
- After transposing the numpy array,
- label became the first row,
- the rest 1:785 columns are pixels

- Thus, data_(dev/train)[0] gives lables
- And, data_(dev/train)[1:785] gives pixels
- Visualizing for better understanding

In [8]:
data_dev #First row is labels

array([[6, 0, 0, ..., 3, 0, 2],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(785, 8400))

In [9]:
Y_dev

array([6, 0, 0, ..., 3, 0, 2], shape=(8400,))

In [10]:
X_dev

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(784, 8400))

In [11]:
data_train #First row is labels

array([[9, 3, 9, ..., 9, 3, 9],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(785, 33600))

In [12]:
Y_train

array([9, 3, 9, ..., 9, 3, 9], shape=(33600,))

In [13]:
X_train

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(784, 33600))

# Neural Net

## Maths behind initializing parameters
- We need to generate random values for the weights and biases for initialization
- So what range to keep to ensure
- No vanishing or exploding gradients/activations
- Control parameters needed:
    - mean = 0
    - controlled variance

### Why mean should be 0?
Let us look at one neuron
  $$z = w_1x_1 + w_2x_2 + \dots + w_nx_n$$
Taking expectation or average behaviour
  $$E[z] = E[w_1]E[x_1] + \dots + E[w_n]E[x_n]$$
If weights are symmetric around 0, that is mean = 0
  $$E[w_i] = 0 \Rightarrow E[z] = 0$$
Thus half neurons positive, half neurons negative

Thus we have a variety in gradients

If mean ≠ 0
  $$E[w_i] \neq 0 \Rightarrow z \text{ shifts in one direction}$$

### Why controlled variance is needed?
Controlled variance implies that we can't have a big range
- Variance of weighted sum:
    $$\text{Var}(z) = n \cdot \text{Var}(w) \cdot \text{Var}(x)$$
- In each layer we will be multipliying variance
  $$\text{Var}(z_L) = k^L \text{Var}(z_0)$$
- So after a few layers we will have $$z \to \infty$$

This is the **vanishing/exploding gradient problem.**

### Why (-0.5, 0.5) Range Chosen?
- Mean = 0
- Small Range
- But also it satisfies the condition:
  $$\text{Var}(w) = \frac{2}{n_{in}} \text{ (ReLU)}$$
- Because we want
  $$\text{Var}(z_{\text{layer}}) \approx \text{Var}(z_{\text{previous}})$$
- Proof :
  - In each layer we multiply gradients, assuming k layers
    $$\text{Var}(z_L) = k^L \text{Var}(z_0)$$
  - We want signal strength to be preserved, that is not vanish or explode, So
    $$\text{Var}(z_{\text{layer}}) \approx \text{Var}(z_{\text{previous}})$$
  - Following constraints need to be met
    | Condition | Outcome |
| :--- | :--- |
| **$k > 1$** | **Explosion:** Variance grows exponentially, leading to numerical instability. |
| **$k < 1$** | **Vanishing:** Variance shrinks to zero; the network stops learning. |
| **$k = 1$** | **Stable Learning:** Signal variance is preserved throughout the network. |
  - Thus, solving for weight variance, we get
    $$\text{Var}(w) = \frac{2}{n_{in}} \text{ (ReLU)}$$
  - This gives **He Initialization**
    $$W \sim \mathcal{N}\left(0, \frac{2}{n_{in}}\right)$$

### Range Analysis
| Range | What happens |
| :--- | :--- |
| **$(0, 1)$** | **Neurons identical:** Symmetry isn't broken; neurons learn the same features. |
| **$(-1, 0)$** | **Neurons dead:** For ReLU, weights start in the "off" zone, causing dead neurons, also known as **dying ReLU problem** |
| **$(-1, 1)$** | **Exploding signals:** The range is too wide for deep architectures without normalization. |
| **$(-0.5, 0.5)$** | **Works shallow only:** Initial signal is okay, but degrades in deeper layers. |
| **He Init** | **Stable deep learning:** Specifically tuned to keep $k \approx 1$ for ReLU. |

## How to initialize parameters
- For input layer , or Layer 0,
- We have 784 neurons, 1st layer has 32 neurons
- Now, each neuron in 0th layer is connected to each neuron in 1st layer

### Weights
- so for that 1 singular neuron we need 32 weights for each of it's connections in the 1st layer
- Similary we have to do it for all the neurons in the 0th layer, which are 784 in total
- So we can generate a random values array of size :
- 32 rows x 784 columns

### Bias
- For each singular neuron in 1st layer, we need to add a singular bias value
- Thus in 1st layer we have 32 neurons so, we need 32 bias values
- So we can generate a random values array of size :
- 32 rows x 1 column

### Layers
- Now for each layer we can repeat this process

### Generalizing
- **Wn = np.random.rand(len(layer n+1), len(layer n))**
- **Bn = np.random.rand(len(layer n+1), 1)**

### Architecture
layers(784,32,16,10)
- 784 neurons in input layer
- 10 neurons in output layer(for each digit 0-9)
- 2 hidden layers
- 1st hidden layer is 32 neurons
- 2nd hidden layer is 16 neurons

In [14]:
#Initializing the starting weights and biases
def init_params():
    #np.random.rand gives value between [0,1] so,
    #[0,1] - 0.5 = [-0.5,0.5]
    W1 = np.random.rand(32,784) - 0.5
    b1 = np.random.rand(32,1) - 0.5
    W2 = np.random.rand(16,32) - 0.5
    b2 = np.random.rand(16,1) - 0.5
    W3 = np.random.rand(10,16) - 0.5
    b3 = np.random.rand(10,1) - 0.5
    return W1,b1,W2,b2,W3,b3 

In [17]:
print(init_params())

(array([[-0.05038225,  0.15962483, -0.02654122, ...,  0.2419119 ,
        -0.35621284, -0.43668102],
       [-0.2244792 ,  0.23698912,  0.24138925, ...,  0.10625698,
         0.13683257,  0.23235965],
       [-0.14571768,  0.16074435, -0.42686602, ...,  0.47394142,
         0.37653439, -0.15256764],
       ...,
       [-0.48779727, -0.07631319,  0.2236077 , ...,  0.30656136,
         0.39413184,  0.07471621],
       [ 0.39337175, -0.19139795,  0.26781633, ..., -0.27953901,
        -0.00268951, -0.149398  ],
       [ 0.45453809, -0.32551148,  0.07547868, ..., -0.49310516,
         0.47572757,  0.28027514]], shape=(32, 784)), array([[-0.3827831 ],
       [ 0.13827877],
       [ 0.44477701],
       [-0.38529727],
       [ 0.07747312],
       [-0.48703321],
       [-0.36413605],
       [ 0.2569476 ],
       [ 0.44979616],
       [ 0.13335811],
       [-0.29514861],
       [ 0.10634971],
       [ 0.1636529 ],
       [ 0.3454648 ],
       [ 0.07658925],
       [-0.00284518],
       [-0.33904

### Utility Functions

In [15]:
#Non Linear Function -> Rectified Linear Unit
def ReLU(Z):
    #Iterator style for each element
    return np.maximum(0,Z)

#To get probability distribution in output layer
def softmax(Z):
    return np.exp(Z)/np.sum(np.exp(Z))

#To one hot encode the labels
def one_hot(Y):
    #Creating matrix for labels Y and init with 0
    matrix_Y = np.zeros((Y.size,Y.max()+1))
    #Setting that particular cell to 1
    matrix_Y[np.arange(Y.size),Y] = 1
    #Taking transpose
    matrix_Y = matrix_Y.T
    return matrix_Y

### Terminologies Used
- ##### Z : pre-activation (raw neuron score before activation)

- ##### **𝐴** :  activation (output after nonlinearity)

- ##### **𝑊** : weights (what we are learning)

- ##### **𝑏** : bias (baseline tendency of neuron)

- ##### **𝑑𝑍** : error signal (how much this neuron caused loss), Also called: delta, upstream gradient, local gradient

- ##### **𝑑𝑊** : how weights should change

- ##### **𝑑𝐴** : error passed to previous layer

- ##### g : activation function

## Mapping Scalars to Matrix
- This is built in continuation to micrograd,
- Where we used scalars to build things
- Now we are using matrices,
- So to bridge the gap in understanding

In micrograd:
 scalar → scalar → scalar → backward

In Neural nets:
 matrices → matrices → matrices → backward

### For One Neuron 
- For one output neuron we have,
  $$z = w_1a_1 + w_2a_2 + \dots + w_na_n$$
- Loss depends on z = $L(z)$
- We need
  $$\frac{\partial L}{\partial w_j}$$
- By chain rule : 
  $$\frac{\partial L}{\partial w_j} = \frac{\partial L}{\partial z} \cdot \frac{\partial z}{\partial w_j}$$
- Now, for one output neuron
  $$z = \sum w_i a_i \Rightarrow \frac{\partial z}{\partial w_j} = a_j$$
  because $z = \dots + w_ja_j + \dots$ , which just gives us $a_j$
- So, by just substituting in Chain rule we get:
  $$\frac{\partial L}{\partial w_j} = \delta \cdot a_j$$
- where
  $$\delta = \frac{\partial L}{\partial z}$$
- Why split into two different parts:
- $\delta$ shows the global gradient(as mentioned in micrograd)
  - the one which has come after flowing backward from the loss function
- $a_j$ shows the local gradient(as mentioned in micrograd)
  - the one which is calculated for the current connection,
  - calculation as shown above
- For colloquial understanding:
  __gradient wrt weight = error × input__

## Math

In [16]:
#Forward Pass
def forward(W1,b1,W2,b2,W3,b3,X):
    Z1 = W1.dot(X)+b1
    A1 = ReLU(Z1)
    Z2 = W2.dot(A1)+b2
    A2 = ReLU(Z2)
    Z3 = W3.dot(A2)+b3
    A3 = softmax(Z3)
    return Z1,A1,Z2,A2,Z3,A3

#Backward Pass
def backward(Z1,A1,Z2,A2,Z3,A3,W2,W3,Y):
    m = Y.size
    one_hot_Y = one_hot(Y)
    #Calculating loss, A2 is prediction , one_hot_Y is label value one hot encoded
    dZ3 = A3 - one_hot_Y
    dW3 = 1/m * DZ3.dot(A3.T)